# 🎓 Interview Trainer Agent
## Powered by IBM Granite (watsonx.ai) + RAG

---

**Problem Statement No. 22** — An Interview Trainer Agent powered by RAG (Retrieval-Augmented Generation), running on IBM Cloud using IBM Granite foundation models.

### What this notebook does:
1. **Ingests** structured interview knowledge (questions, HR guidelines, job role profiles, industry insights)
2. **Embeds** documents using IBM Slate embedding model (watsonx.ai)
3. **Indexes** them into a FAISS vector store for fast retrieval
4. **Retrieves** the most relevant context for each user query
5. **Generates** personalized interview preparation plans using **IBM Granite-13B-Chat**

### Supported Roles:
- Software Engineer | Data Scientist | Product Manager | DevOps Engineer

### Supported Levels:
- Entry | Mid | Senior

---
**IBM Cloud Services Used:**
- IBM watsonx.ai (IBM Granite LLM + IBM Slate Embeddings)
- IBM Cloud Object Storage (optional — for large-scale index persistence)

> **Note:** This notebook runs in STUB mode if IBM credentials are not set. Configure `IBM_CLOUD_API_KEY` and `WATSONX_PROJECT_ID` for full IBM Granite-powered responses.

---
## Section 1 — Environment Setup
Install dependencies and configure IBM Cloud credentials.

In [ ]:
# ============================================================
# Cell 1: Install Required Packages
# ============================================================
# Run this cell once. Restart kernel after installation.

import subprocess, sys

PACKAGES = [
    "ibm-watsonx-ai>=1.0.0",    # IBM watsonx.ai SDK (Granite + Slate)
    "faiss-cpu>=1.7.4",          # FAISS vector store (CPU version for IBM Cloud)
    "sentence-transformers>=2.2.2",  # Fallback embedding model
    "pyyaml>=6.0",               # YAML configuration loader
    "numpy>=1.24",               # Numerical operations
    "pandas>=1.5",               # Data manipulation
    "matplotlib>=3.6",           # Visualizations
    "rich>=13.0",                # Rich terminal output
    "tqdm>=4.65",                # Progress bars
]

print("Installing dependencies...")
for pkg in PACKAGES:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", pkg],
        capture_output=True, text=True
    )
    status = "✅" if result.returncode == 0 else "❌"
    print(f"  {status} {pkg}")

print("\n✅ All packages installed. Restart kernel if this is the first run.")

Installing dependencies...
  ✅ ibm-watsonx-ai>=1.0.0
  ✅ faiss-cpu>=1.7.4


In [ ]:
# ============================================================
# Cell 2: Core Imports and Path Configuration
# ============================================================

import os
import sys
import json
import logging
import warnings
from pathlib import Path
from datetime import datetime
from IPython.display import display, Markdown, HTML

warnings.filterwarnings('ignore')

# ---- Path setup -----------------------------------------------
# Works whether running from /notebooks or project root
NOTEBOOK_DIR = Path().resolve()
if NOTEBOOK_DIR.name == 'notebooks':
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # Ensure all relative paths resolve correctly

# ---- Logging setup --------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('InterviewTrainer')

# ---- Create output directories --------------------------------
for d in ['outputs/sessions', 'outputs/logs', 'data/embeddings/faiss_index']:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"✅ Project root: {PROJECT_ROOT}")
print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ Python: {sys.version.split()[0]}")
print(f"✅ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# ============================================================
# Cell 3: IBM Cloud Credentials Configuration
# ============================================================

IBM_CLOUD_API_KEY   = os.environ.get('IBM_CLOUD_API_KEY', '')
WATSONX_PROJECT_ID  = os.environ.get('WATSONX_PROJECT_ID', '')
IBM_COS_INSTANCE_ID = os.environ.get('IBM_COS_INSTANCE_ID', '')

# --- Credential status check ---
cred_status = [
    ("IBM_CLOUD_API_KEY",   bool(IBM_CLOUD_API_KEY)),
    ("WATSONX_PROJECT_ID",  bool(WATSONX_PROJECT_ID)),
    ("IBM_COS_INSTANCE_ID", bool(IBM_COS_INSTANCE_ID)),
]

print("\n📋 IBM Cloud Credential Status:")
print("-" * 45)
all_set = True
for name, is_set in cred_status:
    icon = '✅' if is_set else '⚠️ '
    status = 'Configured' if is_set else 'NOT SET (stub mode)'
    print(f"  {icon} {name:<25} {status}")
    if not is_set and name != 'IBM_COS_INSTANCE_ID':
        all_set = False
print("-" * 45)

if all_set:
    print("\n🚀 Full IBM Granite mode ACTIVE — all credentials configured!")
else:
    print("\n💡 Running in STUB mode — set IBM_CLOUD_API_KEY and WATSONX_PROJECT_ID")
    print("   for full IBM Granite AI-powered responses via watsonx.ai")

---
## Section 2 — Data Ingestion & Preprocessing
Load structured interview knowledge base and process into retrievable document chunks.

In [ ]:
# ============================================================
# Cell 4: Load and Explore Raw Datasets
# ============================================================

from src.data_processor import DataProcessor

data_processor = DataProcessor(data_dir='data/raw')
documents = data_processor.load_all()

# --- Dataset summary ---
print("\n📚 Knowledge Base Summary")
print("=" * 55)
print(f"  Total documents loaded: {len(documents)}")

# Count by source
from collections import Counter
source_counts = Counter(d.metadata.get('source', 'unknown') for d in documents)
print("\n  Documents by source:")
for src, cnt in sorted(source_counts.items()):
    print(f"    • {src:<30} {cnt:>4} docs")

# Count by role
role_counts = Counter(d.metadata.get('role', '') for d in documents if d.metadata.get('role'))
print("\n  Documents by job role:")
for role, cnt in sorted(role_counts.items()):
    print(f"    • {role:<30} {cnt:>4} docs")

print("\n  Sample document preview:")
print("-" * 55)
sample = documents[0]
print(f"  ID: {sample.doc_id}")
print(f"  Metadata: {sample.metadata}")
print(f"  Content preview: {sample.content[:250]}...")
print("=" * 55)

In [ ]:
# ============================================================
# Cell 5: Explore Raw Interview Questions
# ============================================================

import pandas as pd

with open('data/raw/interview_questions.json', 'r') as f:
    questions_data = json.load(f)

# Flatten all questions into a DataFrame
rows = []
for role, levels in questions_data.get('categories', {}).items():
    for level, qtypes in levels.items():
        for qtype, questions in qtypes.items():
            for q in questions:
                rows.append({
                    'role': role,
                    'level': level,
                    'type': qtype,
                    'id': q.get('id', ''),
                    'category': q.get('category', ''),
                    'difficulty': q.get('difficulty', ''),
                    'question': q.get('question', '')[:80] + '...'
                })

df = pd.DataFrame(rows)
print(f"\n📊 Total Interview Questions in Knowledge Base: {len(df)}")
print("\nBreakdown by Role and Level:")
print(df.groupby(['role', 'level', 'type']).size().reset_index(name='count').to_string(index=False))
print()
display(df.head(10))

---
## Section 3 — Embedding & Vector Store Setup
Generate IBM Slate embeddings and build FAISS index for semantic retrieval.

In [ ]:
# ============================================================
# Cell 6: Initialize RAG Pipeline & Build Vector Index
# ============================================================
# This step:
#   1. Loads all documents from data/raw
#   2. Generates embeddings (IBM Slate or SentenceTransformer fallback)
#   3. Builds and saves a FAISS index to data/embeddings/faiss_index
#
# Subsequent runs will load the saved index (fast!)

from src.rag_pipeline import InterviewRAGPipeline

pipeline = InterviewRAGPipeline(config_path='config/config.yaml')

print("🔧 Initializing Interview Trainer RAG Pipeline...")
print("   (First run builds the index, subsequent runs load from cache)\n")

pipeline.initialize(force_rebuild=False)

status = pipeline.status()
print("\n✅ Pipeline Status:")
print("-" * 45)
for key, val in status.items():
    print(f"  {key:<25} {val}")
print("-" * 45)
print("\n🎉 Pipeline ready for interview preparation!")

In [ ]:
# ============================================================
# Cell 7: Test Vector Retrieval
# ============================================================
# Verify the RAG retrieval is working correctly

test_queries = [
    ("What technical questions should I expect for a mid-level software engineer?", "software_engineer", "mid"),
    ("How to answer behavioral questions using STAR method?", None, None),
    ("What is the IBM interview process like?", None, None),
    ("How do I handle class imbalance in machine learning?", "data_scientist", "mid"),
]

print("🔍 Testing RAG Retrieval Quality\n")
print("=" * 65)

for query, role, level in test_queries:
    context = pipeline.retrieve_context(query=query, role=role, level=level, top_k=2)
    print(f"\n📌 Query: {query}")
    print(f"   Role filter: {role or 'none'} | Level: {level or 'none'}")
    print(f"   Retrieved context preview (first 300 chars):")
    print(f"   {context[:300]}...")
    print("-" * 65)

---
## Section 4 — Interview Preparation Engine
Generate personalized interview prep plans powered by IBM Granite.

In [ ]:
# ============================================================
# Cell 8: Configure Your Profile
# ============================================================
# Modify these values to match the candidate's profile

CANDIDATE_PROFILE = {
    "name": "Arjun Sharma",         # Candidate's name
    "role": "software_engineer",    # Options: software_engineer, data_scientist,
                                     #          product_manager, devops_engineer
    "level": "mid",                  # Options: entry, mid, senior
    "target_company": "IBM",         # Target company (for tailored tips)
    "years_of_experience": 4,        # Years of experience
    "key_skills": ["Python", "Java", "REST APIs", "AWS", "Docker"],
    "resume_highlights": [
        "Built microservices handling 10K RPS",
        "Reduced deployment time by 60% with CI/CD automation",
        "Led a team of 3 engineers on a customer portal project"
    ]
}

# ---- Role info from knowledge base ----------------------------
role_info = pipeline.get_role_info(
    role=CANDIDATE_PROFILE["role"],
    level=CANDIDATE_PROFILE["level"]
)

display(Markdown(f"""
## 👤 Candidate Profile
| Field | Value |
|-------|-------|
| **Name** | {CANDIDATE_PROFILE['name']} |
| **Target Role** | {CANDIDATE_PROFILE['role'].replace('_', ' ').title()} |
| **Experience Level** | {CANDIDATE_PROFILE['level'].title()} |
| **Target Company** | {CANDIDATE_PROFILE['target_company']} |
| **Years of Experience** | {CANDIDATE_PROFILE['years_of_experience']} years |
| **Key Skills** | {', '.join(CANDIDATE_PROFILE['key_skills'])} |
"""))

if role_info.get('level_info'):
    li = role_info['level_info']
    display(Markdown(f"""
## 📋 Role Requirements ({CANDIDATE_PROFILE['role'].replace('_',' ').title()} - {CANDIDATE_PROFILE['level'].title()})
- **Core Skills**: {', '.join(li.get('core_skills', []))}
- **Soft Skills**: {', '.join(li.get('soft_skills', []))}
- **Interview Rounds**: {', '.join(li.get('interview_rounds', []))}
- **Key Companies**: {', '.join(li.get('key_companies', []))}
"""))

In [ ]:
# ============================================================
# Cell 9: Generate Full Interview Preparation Plan
# ============================================================
# IBM Granite generates a comprehensive, personalized prep plan
# based on retrieved context from the knowledge base.

print(f"\n🤖 Generating Interview Preparation Plan for {CANDIDATE_PROFILE['name']}...")
print(f"   Role: {CANDIDATE_PROFILE['role']} | Level: {CANDIDATE_PROFILE['level']}")
print("   Powered by IBM Granite via watsonx.ai + RAG\n")

result = pipeline.generate_prep_plan(
    name=CANDIDATE_PROFILE["name"],
    role=CANDIDATE_PROFILE["role"],
    level=CANDIDATE_PROFILE["level"],
)

print(f"✅ Plan generated | LLM mode: {result['metadata']['llm_mode']} "
      f"| Embedding mode: {result['metadata']['embedding_mode']} "
      f"| Docs indexed: {result['metadata']['documents_in_store']}")

display(Markdown("---"))
display(Markdown(f"# 📋 Interview Preparation Plan for {CANDIDATE_PROFILE['name']}"))
display(Markdown(result["preparation_plan"]))

# ---- Save to file ---
session_file = Path('outputs/sessions') / f"prep_plan_{CANDIDATE_PROFILE['name'].replace(' ', '_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
with open(session_file, 'w', encoding='utf-8') as f:
    f.write(f"# Interview Preparation Plan\n")
    f.write(f"**Candidate:** {CANDIDATE_PROFILE['name']}\n")
    f.write(f"**Role:** {CANDIDATE_PROFILE['role']}  |  **Level:** {CANDIDATE_PROFILE['level']}\n")
    f.write(f"**Generated:** {datetime.now().isoformat()}\n\n---\n\n")
    f.write(result['preparation_plan'])
print(f"\n💾 Session saved: {session_file}")

---
## Section 5 — Question Deep-Dive Mode
Get detailed guidance for specific interview questions.

In [ ]:
# ============================================================
# Cell 10: Deep-Dive on a Specific Question
# ============================================================
# Get IBM Granite-powered guidance on any specific interview question

SPECIFIC_QUESTION = "Design a scalable notification system that can send 10 million notifications per day."

print(f"\n🎯 Getting deep guidance for:\n   '{SPECIFIC_QUESTION}'\n")

guidance_result = pipeline.get_question_guidance(
    name=CANDIDATE_PROFILE["name"],
    role=CANDIDATE_PROFILE["role"],
    level=CANDIDATE_PROFILE["level"],
    question=SPECIFIC_QUESTION,
)

display(Markdown(f"## 🧠 Question Deep-Dive: {SPECIFIC_QUESTION[:60]}..."))
display(Markdown(guidance_result["guidance"]))

print("\n📚 Retrieved Context Preview:")
print(guidance_result["context_retrieved"][:500] + "...")

In [ ]:
# ============================================================
# Cell 11: Behavioral Question Guidance
# ============================================================

BEHAVIORAL_QUESTION = "Tell me about a time you had to make a difficult technical decision under uncertainty."

print(f"\n🎭 Behavioral Question Guidance...")
print(f"   Question: {BEHAVIORAL_QUESTION}\n")

beh_result = pipeline.get_question_guidance(
    name=CANDIDATE_PROFILE["name"],
    role=CANDIDATE_PROFILE["role"],
    level=CANDIDATE_PROFILE["level"],
    question=BEHAVIORAL_QUESTION,
)

display(Markdown("## 🎭 Behavioral Question — STAR Framework Guidance"))
display(Markdown(beh_result["guidance"]))

---
## Section 6 — Answer Evaluator
Submit your practice answer and receive AI-powered feedback with scoring.

In [ ]:
# ============================================================
# Cell 12: Answer Evaluator — Get Scored Feedback
# ============================================================
# Practice answering a question and receive:
#   - Score (1-10) with justification
#   - Strengths and weaknesses
#   - Improved version of your answer
#   - Delivery tips

EVAL_QUESTION = "What is the CAP theorem and how does it influence your system design decisions?"

CANDIDATE_ANSWER = """
CAP theorem says a distributed system can only guarantee two out of three properties:
Consistency, Availability, and Partition Tolerance. In my projects, when we built
the order management system, we chose an AP system (Cassandra) because we needed
the system to stay available even if there were network issues, and eventual
consistency was acceptable for order status updates.
"""

print("📝 Evaluating your practice answer...\n")
print(f"Question: {EVAL_QUESTION}")
print(f"Your Answer: {CANDIDATE_ANSWER.strip()}\n")

eval_result = pipeline.evaluate_candidate_answer(
    name=CANDIDATE_PROFILE["name"],
    role=CANDIDATE_PROFILE["role"],
    level=CANDIDATE_PROFILE["level"],
    question=EVAL_QUESTION,
    candidate_answer=CANDIDATE_ANSWER,
)

display(Markdown("## 📊 Answer Evaluation Report"))
display(Markdown(eval_result["feedback"]))

# Save evaluation
eval_file = Path('outputs/sessions') / f"evaluation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
with open(eval_file, 'w', encoding='utf-8') as f:
    f.write(f"# Answer Evaluation\n")
    f.write(f"**Candidate:** {CANDIDATE_PROFILE['name']}\n")
    f.write(f"**Question:** {EVAL_QUESTION}\n")
    f.write(f"**Candidate Answer:** {CANDIDATE_ANSWER}\n\n---\n\n")
    f.write(eval_result['feedback'])
print(f"\n💾 Evaluation saved: {eval_file}")

---
## Section 7 — Multi-Role & Multi-Level Demonstration
Demonstrate the system across different roles and experience levels.

In [ ]:
# ============================================================
# Cell 13: Multi-Role Preparation Plans
# ============================================================
# Generate preparation plans for different roles/levels in batch

DEMO_PROFILES = [
    {"name": "Priya Nair",       "role": "data_scientist",    "level": "entry"},
    {"name": "Rahul Kumar",      "role": "devops_engineer",   "level": "mid"},
    {"name": "Ananya Reddy",     "role": "product_manager",   "level": "entry"},
    {"name": "Vikram Mehta",     "role": "software_engineer", "level": "senior"},
]

print("\n🚀 Generating preparation plans for all demo profiles...\n")

all_results = []
for profile in DEMO_PROFILES:
    print(f"⚙️  Processing: {profile['name']} ({profile['role']} - {profile['level']})")
    result = pipeline.generate_prep_plan(
        name=profile["name"],
        role=profile["role"],
        level=profile["level"],
    )
    all_results.append({**profile, "result": result})
    print(f"   ✅ Done | LLM: {result['metadata']['llm_mode']}")

print(f"\n✅ All {len(DEMO_PROFILES)} profiles processed.")

# Display one result as sample
sample = all_results[0]
display(Markdown(f"---\n## Sample: {sample['name']} ({sample['role']} - {sample['level']})"))
display(Markdown(sample['result']['preparation_plan'][:1500] + "\n\n_[Truncated for display]_"))

---
## Section 8 — Visualization & Analytics
Visual analytics of the knowledge base and interview preparation metrics.

In [ ]:
# ============================================================
# Cell 14: Knowledge Base Analytics & Visualizations
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import defaultdict

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Interview Trainer Agent — Knowledge Base Analytics', fontsize=15, fontweight='bold', y=1.01)

COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# ---- Plot 1: Documents by Source ----
sources = [d.metadata.get('source', 'unknown') for d in documents]
source_counter = Counter(sources)
ax1 = axes[0, 0]
ax1.bar(source_counter.keys(), source_counter.values(), color=COLORS[:len(source_counter)])
ax1.set_title('Documents by Knowledge Source')
ax1.set_xlabel('Source')
ax1.set_ylabel('Number of Documents')
ax1.tick_params(axis='x', rotation=15)
for i, (k, v) in enumerate(source_counter.items()):
    ax1.text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# ---- Plot 2: Questions by Role ----
q_docs = [d for d in documents if d.metadata.get('source') == 'interview_questions']
role_counter = Counter(d.metadata.get('role', 'other') for d in q_docs)
ax2 = axes[0, 1]
wedges, texts, autotexts = ax2.pie(
    role_counter.values(),
    labels=[r.replace('_', '\n') for r in role_counter.keys()],
    autopct='%1.0f%%',
    colors=COLORS[:len(role_counter)],
    startangle=90
)
ax2.set_title('Interview Questions by Job Role')

# ---- Plot 3: Questions by Type ----
type_counter = Counter(d.metadata.get('question_type', 'other') for d in q_docs)
ax3 = axes[1, 0]
bars = ax3.bar(type_counter.keys(), type_counter.values(),
               color=['#2196F3', '#4CAF50', '#FF9800'])
ax3.set_title('Questions by Type')
ax3.set_xlabel('Question Type')
ax3.set_ylabel('Count')
for bar in bars:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             str(int(bar.get_height())), ha='center', fontweight='bold')

# ---- Plot 4: Preparation Timeline by Role/Level ----
prep_data = {
    'Software Eng.\n(Entry)': 6,
    'Software Eng.\n(Mid)':   8,
    'Software Eng.\n(Senior)': 4,
    'Data Sci.\n(Entry)': 8,
    'Data Sci.\n(Mid)': 6,
    'DevOps\n(Entry)': 6,
    'DevOps\n(Mid)': 6,
    'PM\n(Entry)': 6,
}
ax4 = axes[1, 1]
colors_prep = ['#1f77b4' if 'Eng' in k else '#ff7f0e' if 'Sci' in k else '#2ca02c' if 'DevOps' in k else '#d62728'
               for k in prep_data.keys()]
ax4.barh(list(prep_data.keys()), list(prep_data.values()), color=colors_prep)
ax4.set_title('Recommended Preparation Timeline (Weeks)')
ax4.set_xlabel('Weeks')
ax4.axvline(x=6, color='red', linestyle='--', alpha=0.5, label='Average (6 weeks)')
ax4.legend()

plt.tight_layout()
plt.savefig('outputs/knowledge_base_analytics.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Analytics chart saved to outputs/knowledge_base_analytics.png")

In [ ]:
# ============================================================
# Cell 15: Personalized Preparation Roadmap Visualization
# ============================================================

fig, ax = plt.subplots(figsize=(14, 6))

role = CANDIDATE_PROFILE['role']
level = CANDIDATE_PROFILE['level']
name = CANDIDATE_PROFILE['name']

# Get preparation strategy
role_info = pipeline.get_role_info(role, level)
prep = role_info.get('preparation_strategy', {})
timeline_weeks = prep.get('timeline_weeks', 6)
focus_areas = prep.get('focus_areas', ['Technical Skills', 'Behavioral Prep', 'System Design'])

# Create a Gantt-style roadmap
week_ranges = [
    (f"Week 1-{max(1, timeline_weeks//3)}",    focus_areas[0] if len(focus_areas) > 0 else 'Technical Foundation',  '#1f77b4'),
    (f"Week {timeline_weeks//3+1}-{max(2, 2*timeline_weeks//3)}", focus_areas[1] if len(focus_areas) > 1 else 'Behavioral Prep',       '#ff7f0e'),
    (f"Week {2*timeline_weeks//3+1}-{timeline_weeks}",  focus_areas[2] if len(focus_areas) > 2 else 'Mock Interviews',          '#2ca02c'),
]

for i, (week_range, topic, color) in enumerate(week_ranges):
    ax.barh(i, timeline_weeks // 3, left=i * (timeline_weeks // 3),
            color=color, alpha=0.75, edgecolor='white', height=0.6)
    ax.text(
        i * (timeline_weeks // 3) + (timeline_weeks // 6), i,
        f"{week_range}\n{topic[:45]}",
        ha='center', va='center', fontsize=9, fontweight='bold', color='white'
    )

ax.set_yticks(range(len(week_ranges)))
ax.set_yticklabels(['Phase 1\nFoundation', 'Phase 2\nIntermediate', 'Phase 3\nFinal Prep'])
ax.set_xlabel('Weeks')
ax.set_title(f'Personalized Interview Prep Roadmap — {name}\n({role.replace("_", " ").title()} | {level.title()} Level | {timeline_weeks} Weeks)', fontsize=13)
ax.set_xlim(0, timeline_weeks + 1)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/preparation_roadmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n🗺️  Roadmap saved to outputs/preparation_roadmap.png")

---
## Section 9 — Interactive Session
Full interactive interview simulation session.

In [ ]:
# ============================================================
# Cell 16: Interactive Interview Session Class
# ============================================================

class InterviewSession:
    """
    Manages an interactive interview practice session.
    Tracks question history, scores, and generates a final report.
    """

    def __init__(self, pipeline, candidate_profile: dict):
        self.pipeline = pipeline
        self.profile = candidate_profile
        self.history = []  # list of {question, answer, feedback, score}
        self.session_id = datetime.now().strftime('%Y%m%d_%H%M%S')

    def practice_question(self, question: str, your_answer: str = "") -> str:
        """
        Practice a question:
        - If your_answer provided: evaluate it and return feedback
        - Otherwise: return guidance on how to answer
        """
        if your_answer.strip():
            result = self.pipeline.evaluate_candidate_answer(
                name=self.profile['name'],
                role=self.profile['role'],
                level=self.profile['level'],
                question=question,
                candidate_answer=your_answer,
            )
            self.history.append({
                'question': question,
                'answer': your_answer,
                'feedback': result['feedback'],
                'type': 'evaluation'
            })
            return result['feedback']
        else:
            result = self.pipeline.get_question_guidance(
                name=self.profile['name'],
                role=self.profile['role'],
                level=self.profile['level'],
                question=question,
            )
            self.history.append({
                'question': question,
                'answer': '',
                'feedback': result['guidance'],
                'type': 'guidance'
            })
            return result['guidance']

    def save_session_report(self) -> str:
        """Save the full session as a Markdown report."""
        report_path = Path('outputs/sessions') / f"session_{self.session_id}.md"
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(f"# Interview Practice Session Report\n")
            f.write(f"**Candidate:** {self.profile['name']}\n")
            f.write(f"**Role:** {self.profile['role'].replace('_',' ').title()} | "
                    f"**Level:** {self.profile['level'].title()}\n")
            f.write(f"**Session ID:** {self.session_id}\n")
            f.write(f"**Questions Practiced:** {len(self.history)}\n\n---\n\n")

            for i, item in enumerate(self.history, 1):
                f.write(f"## Q{i}: {item['question']}\n\n")
                if item['answer']:
                    f.write(f"**Your Answer:**\n{item['answer']}\n\n")
                f.write(f"**{'Feedback' if item['type'] == 'evaluation' else 'Guidance'}:**\n")
                f.write(f"{item['feedback']}\n\n---\n\n")

        return str(report_path)


# Initialize session
session = InterviewSession(pipeline, CANDIDATE_PROFILE)
print(f"✅ Interactive session initialized for {CANDIDATE_PROFILE['name']}")
print(f"   Role: {CANDIDATE_PROFILE['role']} | Level: {CANDIDATE_PROFILE['level']}")
print("\nUsage:")
print("  session.practice_question('Your question here')           # Get guidance")
print("  session.practice_question('Question', 'Your answer...')   # Get evaluated")

In [ ]:
# ============================================================
# Cell 17: Run Interactive Session Demo
# ============================================================

print("\n=" * 60)
print("🎙️  INTERVIEW PRACTICE SESSION")
print("=" * 60)

# --- Practice round 1: Get guidance ---
q1 = "Explain how you would design a rate limiter for a public API."
print(f"\n📌 Question 1 (Guidance mode):\n   {q1}\n")
guidance1 = session.practice_question(q1)
display(Markdown(guidance1[:1200] + "\n\n_[See full guidance above]_"))

print("\n" + "-" * 60)

# --- Practice round 2: Evaluate an answer ---
q2 = "How do you ensure code quality in your team?"
my_answer = """
I enforce code quality through a multi-layered approach. First, we have mandatory code reviews 
with at least one senior engineer's approval. We use SonarQube for static analysis and enforce 
80% test coverage as a CI gate. I introduced coding standards documentation and weekly knowledge 
sharing sessions. We also do pair programming for complex features. This reduced production bugs 
by 35% over 6 months.
"""

print(f"📌 Question 2 (Evaluation mode):\n   {q2}")
print(f"\n   Your answer: {my_answer.strip()[:200]}...\n")
feedback2 = session.practice_question(q2, my_answer)
display(Markdown("### 📊 Evaluation Feedback:"))
display(Markdown(feedback2))

# --- Save session report ---
report_path = session.save_session_report()
print(f"\n💾 Session report saved: {report_path}")
print(f"   Total questions practiced: {len(session.history)}")

---
## Section 10 — IBM watsonx.ai Direct API Demo
Direct demonstration of IBM Granite model capabilities.

In [ ]:
# ============================================================
# Cell 18: IBM watsonx.ai Direct API Call Demo
# ============================================================
# This cell demonstrates a direct call to IBM Granite
# via the watsonx.ai SDK — separate from the RAG pipeline.

def demo_granite_direct(prompt: str, api_key: str = '', project_id: str = '',
                         url: str = 'https://us-south.ml.cloud.ibm.com') -> str:
    """
    Direct IBM Granite inference call.
    Returns generated text or a stub message if credentials are absent.
    """
    if not api_key or api_key.startswith('MISSING'):
        return (
            "**[STUB]** IBM Granite response would appear here.\n\n"
            "To enable: set IBM_CLOUD_API_KEY and WATSONX_PROJECT_ID env vars.\n\n"
            f"Prompt sent: `{prompt[:200]}...`"
        )

    try:
        from ibm_watsonx_ai import Credentials
        from ibm_watsonx_ai.foundation_models import ModelInference
        from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams

        credentials = Credentials(api_key=api_key, url=url)
        model = ModelInference(
            model_id="ibm/granite-13b-chat-v2",
            credentials=credentials,
            project_id=project_id,
            params={
                GenParams.MAX_NEW_TOKENS: 512,
                GenParams.TEMPERATURE: 0.7,
                GenParams.DECODING_METHOD: "sample",
            }
        )
        return model.generate_text(prompt=prompt)
    except Exception as e:
        return f"Error calling IBM Granite: {e}"


# Demo prompt
demo_prompt = """
You are an expert interview coach. A candidate is applying for a Senior Software Engineer
position at IBM. Give them 3 specific tips for standing out in the technical interview,
focusing on IBM's values and technology stack (IBM Cloud, watsonx.ai, Red Hat OpenShift).
"""

print("🤖 Direct IBM Granite API Demo")
print("=" * 55)
print(f"Model: ibm/granite-13b-chat-v2")
print(f"Endpoint: {os.environ.get('IBM_CLOUD_API_KEY', 'not-set')[:10]}...")
print("\nPrompt:")
print(demo_prompt.strip())
print("\nResponse:")
print("-" * 55)

response = demo_granite_direct(
    prompt=demo_prompt,
    api_key=IBM_CLOUD_API_KEY,
    project_id=WATSONX_PROJECT_ID,
)

display(Markdown(response))

---
## Section 11 — Project Summary & Deployment Guide

In [ ]:
# ============================================================
# Cell 19: Project Summary Report
# ============================================================

status = pipeline.status()

summary_md = f"""
# 🎓 Interview Trainer Agent — Project Summary

## System Overview
| Component | Details |
|-----------|--------|
| **Project** | Interview Trainer Agent (Problem Statement No. 22) |
| **Architecture** | RAG (Retrieval-Augmented Generation) |
| **LLM** | IBM Granite-13B-Chat-v2 (watsonx.ai) |
| **Embeddings** | IBM Slate-125M-English-RTRVR |
| **Vector Store** | FAISS (IndexFlatIP — Cosine Similarity) |
| **Platform** | IBM Cloud / IBM Watson Studio |

## Knowledge Base Statistics
| Metric | Value |
|--------|-------|
| Total Documents Indexed | {status['documents_indexed']} |
| Supported Roles | 4 (Software Engineer, Data Scientist, PM, DevOps) |
| Supported Levels | 3 (Entry, Mid, Senior) |
| Question Types | Technical, Behavioral, HR |
| Data Sources | Interview Questions, HR Guidelines, Job Roles, Industry Insights |

## Features Demonstrated
- ✅ Personalized interview question set generation
- ✅ Deep-dive guidance on specific questions
- ✅ Answer evaluation with scoring and improvement tips
- ✅ Multi-role, multi-level preparation support
- ✅ Visual preparation roadmap
- ✅ Session management with saved reports
- ✅ Direct IBM Granite API integration
- ✅ IBM Slate semantic embeddings
- ✅ Knowledge base analytics

## IBM Cloud Services Used
| Service | Purpose |
|---------|--------|
| IBM watsonx.ai | LLM inference (IBM Granite) + Embeddings (IBM Slate) |
| IBM Watson Studio | Jupyter notebook hosting and execution |
| IBM Cloud Object Storage | Index and session data persistence (optional) |

## Current Session Modes
| Component | Mode |
|-----------|------|
| Embedding Mode | `{status['embedding_mode']}` |
| LLM Mode | `{status['llm_mode']}` |
"""

display(Markdown(summary_md))

# Save summary
with open('outputs/project_summary.md', 'w', encoding='utf-8') as f:
    f.write(summary_md)
print("\n💾 Project summary saved to outputs/project_summary.md")

---

## 🎯 Next Steps

1. **Configure IBM Credentials**: Set `IBM_CLOUD_API_KEY` and `WATSONX_PROJECT_ID` in your Watson Studio environment to enable full IBM Granite-powered responses.

2. **Expand Knowledge Base**: Add more questions to `data/raw/interview_questions.json` and rebuild the index with `pipeline.initialize(force_rebuild=True)`.

3. **Deploy as an App**: Use `notebooks/` as a Watson Studio Space deployment or export to IBM Code Engine.

4. **Add Resume Parsing**: Extend the pipeline with `pdfminer` or `python-docx` to parse uploaded resumes for more personalized prep.

---
*Interview Trainer Agent v1.0 — Built with IBM Granite + watsonx.ai*